# Reproduce manuscript tables and figures

This notebook is a guided wrapper around the maintained Python scripts for *Equilibrium Data Coverage and Representation in Weather-Resolved Direct Air Capture Analysis*. It uses the bundled compressed inputs and does not contain a second scientific implementation.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
DATA = REPO / 'data' / 'frozen'
WEATHER = REPO / 'data' / 'weather' / 'MAIN_climate_hourly_2015_2024.csv'
RESULTS = REPO / 'results'
sys.path.insert(0, str(REPO / 'src'))
print('repository:', REPO)

## 1. Materialize the bundled compressed inputs

In [ ]:
!{sys.executable} {REPO / 'scripts' / 'materialize_data.py'}

## 2. Run the unit and regression tests

In [ ]:
!{sys.executable} -m pytest {REPO / 'tests'}

## 3. Regenerate coverage, H, H/N, Table S6 and pairwise tables

The `all` target follows the raw-input weather to H to H/N path and also reconstructs the retained IRA pair classifications.

In [ ]:
!{sys.executable} {REPO / 'scripts' / 'reproduce_tables.py'} --input-dir {DATA} --output-dir {RESULTS / 'tables'} --target all

## 4. Replot main Figures 2-7 and SI Figures S1-S7

These are numerical source-data checks, not artwork-identical regressions. Figure 1 is handled by the exact flowchart command below.

In [ ]:
!{sys.executable} {REPO / 'scripts' / 'reproduce_figures.py'} --input-dir {DATA} --weather-file {WEATHER} --output-dir {RESULTS / 'figures'}
!{sys.executable} {REPO / 'scripts' / 'reproduce_si_figures.py'} --input-dir {DATA} --output-dir {RESULTS / 'si_figures'}
!{sys.executable} {REPO / 'scripts' / 'reproduce_flowchart_exact.py'} --output {RESULTS / 'measurement_to_co2_release_flowchart.png'}

## 5. Verify the completed PADAC solar-regeneration archive

This verifies all 36 retained city results and the 13-of-435 solar-regeneration claim. A full CPU-intensive rerun is available through `reproduce_padac_all.py --workers 2 --nz 20`.

In [ ]:
!{sys.executable} {REPO / 'scripts' / 'verify_padac_solar.py'} --result-dir {REPO / 'validation' / 'padac_partial'} --output {RESULTS / 'padac' / 'solar_pair_verification.json'}

## 6. Inspect generated regression reports

In [ ]:
from IPython.display import display
import pandas as pd

for report in sorted((RESULTS / 'tables').glob('*regression.csv')):
    print(report.name)
    display(pd.read_csv(report))